<a href="https://colab.research.google.com/github/MirunaLivanov/Veridion-Project/blob/main/content/02-getting-started/jupyter_notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veridion Project


In [2]:
!pip install rapidfuzz
import pandas as pd
from rapidfuzz import fuzz
df = pd.read_csv("presales_data_sample.csv")

df = df.drop_duplicates(subset=['input_row_key', 'veridion_id'])


def compute_score(row):
    score = 0

    input_name = str(row['input_company_name'])
    candidate_name = str(row['company_name'])

    name_score = fuzz.token_set_ratio(input_name, candidate_name) / 100
    score += name_score * 0.6

    if row['input_main_country_code'] == row['main_country_code']:
        score += 0.2

    if str(row['input_main_city']).lower() == str(row['main_city']).lower():
        score += 0.1

    if pd.notnull(row.get('website_domain', None)):
        score += 0.1

    return score


df['score'] = df.apply(compute_score, axis=1)

grouped = df.groupby('input_row_key')

results = []

for key, group in grouped:
    best = group.loc[group['score'].idxmax()].copy()

    if best['score'] < 0.55:
        best['company_name'] = "NO MATCH"
        best['veridion_id'] = None

    results.append(best)

clean_df_raw = pd.DataFrame(results)


def assign_confidence(score):
    if score >= 0.75:
        return 'HIGH'
    elif score >= 0.55:
        return 'MEDIUM'
    else:
        return 'LOW'

clean_df_raw['confidence'] = clean_df_raw['score'].apply(assign_confidence)

clean_df = clean_df_raw[[
    'input_row_key',
    'input_company_name',
    'company_name',
    'veridion_id',
    'score',
    'confidence',
    'input_main_country',
    'main_country',
    'main_city',
    'website_domain'
]]


print("\nConfidence distribution:")
print(clean_df['confidence'].value_counts())

clean_df.to_csv("cleaned_suppliers.csv", index=False)
clean_df.to_excel("cleaned_suppliers.xlsx", index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.3 MB/s eta 0:00:00

Confidence distribution:
confidence
LOW       243
HIGH      182
MEDIUM    167
Name: count, dtype: int64


In [ ]:
import pandas as pd
from rapidfuzz import fuzz
df = pd.read_csv("presales_data_sample.csv")
df = df.drop_duplicates(subset=['input_row_key', 'veridion_id'])

THRESHOLD = 0.55

def compute_score(row):
    score = 0

    input_name = str(row['input_company_name'])
    candidate_name = str(row['company_name'])


    name_score = fuzz.token_set_ratio(input_name, candidate_name) / 100
    score += name_score * 0.65


    if row.get('input_main_country_code') == row.get('main_country_code'):
        score += 0.20


    if str(row.get('input_main_city', '')).lower() == str(row.get('main_city', '')).lower():
        score += 0.10


    if pd.notnull(row.get('website_domain', None)):
        score += 0.05

    return score


df['score'] = df.apply(compute_score, axis=1)
grouped = df.groupby('input_row_key')

results = []

for key, group in grouped:

    best_idx = group['score'].idxmax()
    best = group.loc[best_idx]

    base_record = {
        'input_row_key': key,
        'input_company_name': group.iloc[0]['input_company_name'],
        'score': best['score']
    }
    if best['score'] < THRESHOLD:
        base_record.update({
            'company_name': "NO MATCH",
            'veridion_id': None,
            'confidence': "NONE"
        })
    else:
        base_record.update({
            'company_name': best['company_name'],
            'veridion_id': best['veridion_id'],
            'confidence': (
                "HIGH" if best['score'] >= 0.80 else
                "MEDIUM" if best['score'] >= 0.65 else
                "LOW"
            )
        })

    results.append(base_record)

clean_df = pd.DataFrame(results)

clean_df = clean_df.sort_values('score', ascending=False)
clean_df = clean_df.drop_duplicates(subset=['veridion_id'])

clean_df = clean_df[[
    'input_row_key',
    'input_company_name',
    'company_name',
    'veridion_id',
    'score',
    'confidence'
]]





clean_df.to_csv("cleaned_suppliers2.csv", index=False)
clean_df.to_excel("cleaned_suppliers2.xlsx", index=False)






print("\nENTITY RESOLUTION SUMMARY")
print("Total input entities:", len(clean_df))
print(clean_df['confidence'].value_counts())


ENTITY RESOLUTION SUMMARY
Total input entities: 331
confidence
HIGH      157
MEDIUM    114
LOW        59
NONE        1
Name: count, dtype: int64
